# Entrega 3 - Design de Experimentos

Este notebook prepara o Design de Experimentos (DoE) das fases CRISP-DM de Modelagem e Avaliacao. A etapa segue a hierarquia solicitada no enunciado:

`Features -> Modelo -> Hiperparametros -> Metricas de Avaliacao`

Atencao: o objetivo desta entrega e preparar o DoE, nao executar os modelos. Portanto, este notebook define os objetos, tabelas e configuracoes necessarias para a execucao futura, mas nao chama `fit()` em nenhum estimador ou busca de hiperparametros.

## 1. Carregamento dos Artefatos da Entrega 2

A base de treino e teste ja foi separada e transformada na Entrega 2. Nesta etapa, esses arquivos sao usados apenas para identificar colunas disponiveis, dimensoes e distribuicao da classe alvo.

In [1]:
from itertools import product
from pathlib import Path

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

pd.set_option("display.max_columns", 120)

project_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
outputs_dir = project_dir / "outputs"

train_path = outputs_dir / "drug_events_train_transformed.csv"
test_path = outputs_dir / "drug_events_test_transformed.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

target_column = "serious"
id_columns = ["Unnamed: 0", "safetyreportid"]

dataset_summary = pd.DataFrame(
    {
        "conjunto": ["treino", "teste"],
        "registros": [len(train_df), len(test_df)],
        "atributos": [train_df.shape[1], test_df.shape[1]],
    }
)

class_distribution = (
    train_df[target_column]
    .value_counts(normalize=True)
    .rename_axis("classe")
    .reset_index(name="proporcao_treino")
)
class_distribution["proporcao_treino"] = class_distribution["proporcao_treino"].round(4)

display(dataset_summary)
display(class_distribution)

,conjunto,registros,atributos
0,treino,22616,16
1,teste,9693,16


,classe,proporcao_treino
0,1,0.623
1,2,0.377


## 2. Subconjuntos de Atributos

Os subconjuntos foram definidos de forma incremental. Assim, o experimento permite medir o efeito da adicao de atributos de medicamentos e textos agregados sobre o desempenho dos modelos.

In [2]:
numeric_features = [
    "patient.drug.count",
    "feature_n_active_substances",
    "feature_n_drug_types",
]

categorical_features = [
    "occurcountry",
    "patient.patientsex",
    "patient.ageGroupCalculated",
]

binary_features = [
    "patient.ageGroupCalculated_was_imputed",
    "feature_has_medicinal_product",
    "feature_multiple_drugs",
    "feature_is_usa",
]

text_features = [
    "patient.drug.activesubstance.activesubstancename",
    "patient.drug.medicinalproduct",
]

feature_sets = {
    "F1_basico_clinico": {
        "description": "Atributos demograficos, pais de ocorrencia, faixa etaria imputada e indicador de imputacao.",
        "columns": [
            "occurcountry",
            "patient.patientsex",
            "patient.ageGroupCalculated",
            "patient.ageGroupCalculated_was_imputed",
            "feature_is_usa",
        ],
    },
    "F2_medicamentos_estruturado": {
        "description": "F1 acrescido de contagens e indicadores estruturados de medicamentos.",
        "columns": [
            "occurcountry",
            "patient.patientsex",
            "patient.ageGroupCalculated",
            "patient.ageGroupCalculated_was_imputed",
            "feature_is_usa",
            "patient.drug.count",
            "feature_n_active_substances",
            "feature_n_drug_types",
            "feature_has_medicinal_product",
            "feature_multiple_drugs",
        ],
    },
    "F3_completo_com_texto": {
        "description": "F2 acrescido dos textos agregados de substancia ativa e produto medicinal.",
        "columns": [
            "occurcountry",
            "patient.patientsex",
            "patient.ageGroupCalculated",
            "patient.ageGroupCalculated_was_imputed",
            "feature_is_usa",
            "patient.drug.count",
            "feature_n_active_substances",
            "feature_n_drug_types",
            "feature_has_medicinal_product",
            "feature_multiple_drugs",
            "patient.drug.activesubstance.activesubstancename",
            "patient.drug.medicinalproduct",
        ],
    },
}

feature_sets_table = pd.DataFrame(
    [
        {
            "feature_set": name,
            "descricao": spec["description"],
            "n_atributos": len(spec["columns"]),
            "atributos": ", ".join(spec["columns"]),
        }
        for name, spec in feature_sets.items()
    ]
)

feature_sets_table

,feature_set,descricao,n_atributos,atributos
0,F1_basico_clinico,"Atributos demograficos, pais de ocorrencia, fa...",5,"occurcountry, patient.patientsex, patient.ageG..."
1,F2_medicamentos_estruturado,F1 acrescido de contagens e indicadores estrut...,10,"occurcountry, patient.patientsex, patient.ageG..."
2,F3_completo_com_texto,F2 acrescido dos textos agregados de substanci...,12,"occurcountry, patient.patientsex, patient.ageG..."


## 3. Pre-processamento Planejado

Os pipelines abaixo serao usados na etapa futura de execucao. Eles preservam a separacao treino/teste e evitam vazamento porque todo pre-processamento aprendido sera ajustado dentro da validacao cruzada.

In [3]:
def build_preprocessor(selected_columns):
    selected_numeric = [c for c in numeric_features if c in selected_columns]
    selected_categorical = [c for c in categorical_features if c in selected_columns]
    selected_binary = [c for c in binary_features if c in selected_columns]
    selected_text = [c for c in text_features if c in selected_columns]

    transformers = []

    if selected_numeric:
        transformers.append(
            (
                "numeric",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                selected_numeric,
            )
        )

    if selected_categorical:
        transformers.append(
            (
                "categorical",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("encoder", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                selected_categorical,
            )
        )

    if selected_binary:
        transformers.append(("binary", "passthrough", selected_binary))

    for text_column in selected_text:
        transformers.append(
            (
                f"text_{text_column}",
                CountVectorizer(max_features=300, min_df=5, binary=True),
                text_column,
            )
        )

    return ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.0)

## 4. Modelos e Hiperparametros

Cada arquitetura sera avaliada dentro de uma busca de hiperparametros. Espacos pequenos foram associados a `GridSearchCV`; espacos maiores, a `RandomizedSearchCV`.

In [4]:
model_specs = {
    "regressao_logistica": {
        "estimator": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
        "search": "GridSearchCV",
        "param_grid": {
            "model__C": [0.1, 1.0, 10.0],
            "model__penalty": ["l2"],
            "model__solver": ["lbfgs"],
        },
    },
    "arvore_decisao": {
        "estimator": DecisionTreeClassifier(class_weight="balanced", random_state=42),
        "search": "GridSearchCV",
        "param_grid": {
            "model__max_depth": [3, 5, 10, None],
            "model__min_samples_leaf": [1, 5, 10],
            "model__criterion": ["gini", "entropy"],
        },
    },
    "random_forest": {
        "estimator": RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1),
        "search": "RandomizedSearchCV",
        "param_grid": {
            "model__n_estimators": [100, 200, 300],
            "model__max_depth": [5, 10, 20, None],
            "model__min_samples_leaf": [1, 5, 10],
            "model__max_features": ["sqrt", "log2"],
        },
        "n_iter": 20,
    },
    "gradient_boosting": {
        "estimator": GradientBoostingClassifier(random_state=42),
        "search": "RandomizedSearchCV",
        "param_grid": {
            "model__n_estimators": [100, 200, 300],
            "model__learning_rate": [0.03, 0.05, 0.1],
            "model__max_depth": [2, 3, 5],
            "model__subsample": [0.8, 1.0],
        },
        "n_iter": 20,
    },
    "svm": {
        "estimator": SVC(class_weight="balanced", probability=True, random_state=42),
        "search": "GridSearchCV",
        "param_grid": {
            "model__C": [0.1, 1.0, 10.0],
            "model__kernel": ["linear", "rbf"],
            "model__gamma": ["scale", "auto"],
        },
    },
}

models_table = pd.DataFrame(
    [
        {
            "modelo": name,
            "estimador": spec["estimator"].__class__.__name__,
            "busca": spec["search"],
            "hiperparametros": ", ".join(spec["param_grid"].keys()),
        }
        for name, spec in model_specs.items()
    ]
)

models_table

,modelo,estimador,busca,hiperparametros
0,regressao_logistica,LogisticRegression,GridSearchCV,"model__C, model__penalty, model__solver"
1,arvore_decisao,DecisionTreeClassifier,GridSearchCV,"model__max_depth, model__min_samples_leaf, mod..."
2,random_forest,RandomForestClassifier,RandomizedSearchCV,"model__n_estimators, model__max_depth, model__..."
3,gradient_boosting,GradientBoostingClassifier,RandomizedSearchCV,"model__n_estimators, model__learning_rate, mod..."
4,svm,SVC,GridSearchCV,"model__C, model__kernel, model__gamma"


## 5. Metricas e Validacao Cruzada

Como a variavel alvo apresenta desbalanceamento moderado, a metrica primaria planejada e `f1_macro`. Tambem serao registradas metricas complementares para treino, validacao cruzada e teste.

In [5]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    "roc_auc": "roc_auc",
}

primary_metric = "f1_macro"

metrics_table = pd.DataFrame(
    [
        {"metrica": name, "uso": "selecao" if name == primary_metric else "diagnostico"}
        for name in scoring
    ]
)

metrics_table

,metrica,uso
0,accuracy,diagnostico
1,balanced_accuracy,diagnostico
2,precision_macro,diagnostico
3,recall_macro,diagnostico
4,f1_macro,selecao
5,roc_auc,diagnostico


## 6. Blueprint Experimental

A tabela abaixo materializa a hierarquia `Features -> Modelo -> Hiperparametros -> Metricas`. Ela define todas as combinacoes planejadas para a etapa futura de execucao.

In [6]:
experiment_blueprint = []

for feature_set_name, model_name in product(feature_sets.keys(), model_specs.keys()):
    feature_spec = feature_sets[feature_set_name]
    model_spec = model_specs[model_name]
    pipeline = Pipeline(
        steps=[
            ("preprocess", build_preprocessor(feature_spec["columns"])),
            ("model", model_spec["estimator"]),
        ]
    )

    experiment_blueprint.append(
        {
            "feature_set": feature_set_name,
            "modelo": model_name,
            "pipeline": pipeline,
            "tipo_busca": model_spec["search"],
            "parametros": model_spec["param_grid"],
            "n_iter": model_spec.get("n_iter"),
            "cv": cv_strategy,
            "scoring": scoring,
            "refit": primary_metric,
        }
    )

blueprint_table = pd.DataFrame(
    [
        {
            "feature_set": item["feature_set"],
            "modelo": item["modelo"],
            "tipo_busca": item["tipo_busca"],
            "n_parametros": len(item["parametros"]),
            "metrica_refit": item["refit"],
        }
        for item in experiment_blueprint
    ]
)

blueprint_table

,feature_set,modelo,tipo_busca,n_parametros,metrica_refit
0,F1_basico_clinico,regressao_logistica,GridSearchCV,3,f1_macro
1,F1_basico_clinico,arvore_decisao,GridSearchCV,3,f1_macro
2,F1_basico_clinico,random_forest,RandomizedSearchCV,4,f1_macro
3,F1_basico_clinico,gradient_boosting,RandomizedSearchCV,4,f1_macro
4,F1_basico_clinico,svm,GridSearchCV,3,f1_macro
5,F2_medicamentos_estruturado,regressao_logistica,GridSearchCV,3,f1_macro
6,F2_medicamentos_estruturado,arvore_decisao,GridSearchCV,3,f1_macro
7,F2_medicamentos_estruturado,random_forest,RandomizedSearchCV,4,f1_macro
8,F2_medicamentos_estruturado,gradient_boosting,RandomizedSearchCV,4,f1_macro
9,F2_medicamentos_estruturado,svm,GridSearchCV,3,f1_macro


## 7. Estrutura Futura de Busca

A funcao abaixo demonstra como a busca sera instanciada na proxima etapa. Ela apenas cria o objeto `GridSearchCV` ou `RandomizedSearchCV`; nao executa treinamento.

In [7]:
def build_search_object(experiment):
    common_kwargs = {
        "estimator": experiment["pipeline"],
        "scoring": experiment["scoring"],
        "refit": experiment["refit"],
        "cv": experiment["cv"],
        "return_train_score": True,
        "n_jobs": -1,
    }

    if experiment["tipo_busca"] == "GridSearchCV":
        return GridSearchCV(param_grid=experiment["parametros"], **common_kwargs)

    return RandomizedSearchCV(
        param_distributions=experiment["parametros"],
        n_iter=experiment["n_iter"],
        random_state=42,
        **common_kwargs,
    )

planned_searches = [
    {
        "feature_set": experiment["feature_set"],
        "modelo": experiment["modelo"],
        "search_object": build_search_object(experiment),
    }
    for experiment in experiment_blueprint
]

pd.DataFrame(
    [
        {
            "feature_set": item["feature_set"],
            "modelo": item["modelo"],
            "classe_busca": item["search_object"].__class__.__name__,
        }
        for item in planned_searches
    ]
)

,feature_set,modelo,classe_busca
0,F1_basico_clinico,regressao_logistica,GridSearchCV
1,F1_basico_clinico,arvore_decisao,GridSearchCV
2,F1_basico_clinico,random_forest,RandomizedSearchCV
3,F1_basico_clinico,gradient_boosting,RandomizedSearchCV
4,F1_basico_clinico,svm,GridSearchCV
5,F2_medicamentos_estruturado,regressao_logistica,GridSearchCV
6,F2_medicamentos_estruturado,arvore_decisao,GridSearchCV
7,F2_medicamentos_estruturado,random_forest,RandomizedSearchCV
8,F2_medicamentos_estruturado,gradient_boosting,RandomizedSearchCV
9,F2_medicamentos_estruturado,svm,GridSearchCV


## 8. Modelo de Tabela de Resultados

A execucao futura devera preencher uma linha por combinacao de feature set, modelo e melhor configuracao de hiperparametros.

In [8]:
result_columns = [
    "feature_set",
    "modelo",
    "tipo_busca",
    "melhores_hiperparametros",
    "train_accuracy",
    "train_balanced_accuracy",
    "train_precision_macro",
    "train_recall_macro",
    "train_f1_macro",
    "train_roc_auc",
    "cv_accuracy_mean",
    "cv_accuracy_std",
    "cv_balanced_accuracy_mean",
    "cv_balanced_accuracy_std",
    "cv_precision_macro_mean",
    "cv_precision_macro_std",
    "cv_recall_macro_mean",
    "cv_recall_macro_std",
    "cv_f1_macro_mean",
    "cv_f1_macro_std",
    "cv_roc_auc_mean",
    "cv_roc_auc_std",
    "test_accuracy",
    "test_balanced_accuracy",
    "test_precision_macro",
    "test_recall_macro",
    "test_f1_macro",
    "test_roc_auc",
]

results_template = pd.DataFrame(columns=result_columns)
results_template

,feature_set,modelo,tipo_busca,melhores_hiperparametros,train_accuracy,train_balanced_accuracy,train_precision_macro,train_recall_macro,train_f1_macro,train_roc_auc,cv_accuracy_mean,cv_accuracy_std,cv_balanced_accuracy_mean,cv_balanced_accuracy_std,cv_precision_macro_mean,cv_precision_macro_std,cv_recall_macro_mean,cv_recall_macro_std,cv_f1_macro_mean,cv_f1_macro_std,cv_roc_auc_mean,cv_roc_auc_std,test_accuracy,test_balanced_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_roc_auc


## 9. Criterio de Selecao Final

A configuracao final sera escolhida pela maior media de `f1_macro` na validacao cruzada. Em caso de empate pratico, serao considerados, nesta ordem: menor variancia em `f1_macro`, maior `balanced_accuracy`, menor complexidade do modelo e desempenho consistente no conjunto de teste.

Este notebook encerra a Entrega 3 no nivel de planejamento. A execucao das buscas e o preenchimento das metricas devem ocorrer em uma etapa posterior.